# Job Advertisement Analysis Pipeline
## Part 1 — Data Preparation & Preprocessing

**Author:** Maria Kokkoni
**Purpose:** Data exploration, verification of label alignment, and validation of the preprocessing pipeline before full execution.  
**Script:** All logic shown here is implemented in `preprocessing.py`

---

### Pipeline Overview
This notebook covers the first stage of a two-part NLP pipeline:

| Stage | Task | Owner                                      |
|-------|------|--------------------------------------------|
| 1 | Zone Identification — BERT token classification | Maria Kokkoni (data) + Veseli Shpëtim (training) |
| 2 | Skill Extraction — LLM (Gemini) on identified zones | Schmied Céline                                   |

The notebook is structured into 5 sections:
1. Dataset Exploration
2. Tokenization
3. Label Alignment Verification
4. Sliding Window Verification
5. Full Pipeline Sanity Check


---
## Section 1 — Dataset Exploration

### 1.1 Load the Dataset

The dataset is provided in `annotated.json`. Each entry contains:
- `data['content_clean']` — the raw job advertisement text
- `annotations[0]['result']` — a list of labeled spans with character offsets and zone labels


In [ ]:
import json
from collections import Counter

with open('annotated.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Total entries loaded: {len(data)}')

Total entries loaded: 2699


### 1.2 Inspect a Single Entry

We use entry ID 4163 throughout this section — it is short enough to fully inspect but contains all key structural elements:
- Multiple labeled spans
- The same label appearing twice non-contiguously
- Unlabeled gaps between spans (`\n\n`)


In [ ]:
entry = next(item for item in data if item['id'] == 4163)

text  = entry['data']['content_clean']
spans = entry['annotations'][0]['result']

print(f'Text ({len(text)} characters):')
print(repr(text))
print(f'\nNumber of annotation spans: {len(spans)}')

Text (231 characters):
'Lernende/r Gebäudetechnikplaner/in Lüftung EFZ\n\nDie Hälg Group ist ein Schweizer Familienunternehmen. Wir planen und realisieren Gebäudetechnikprojekte und bieten integrales Facility Management. ...\n\nHälg Group (Lehrstellen)\n\nBasel'

Number of annotation spans: 3


### 1.3 Verify Character Offsets

A key finding from exploration: `value['text']` contains escaped newlines (`\\n`) rather than real newlines, causing string mismatches when compared to `content_clean`. However, `value['start']` and `value['end']` are accurate. The pipeline uses only the offsets — never `value['text']`.


In [ ]:
print('Verifying character offsets:\n')
for span in spans:
    v         = span['value']
    extracted = text[v['start']:v['end']]
    matches   = extracted == v['text']
    print(f"Label: {v['labels'][0]}")
    print(f"  Start: {v['start']}, End: {v['end']}")
    print(f"  Extracted start: {repr(extracted[:15])}")
    print(f"  Extracted end:   {repr(extracted[-15:])}")
    print(f"  Annotation end:  {repr(v['text'][-15:])}")
    print(f"  Offset reliable: {v['start'] < v['end']}")
    print()

print(f'Gap 1 (chars 46-48): {repr(text[46:48])}')
print(f'Gap 2 (chars 198-200): {repr(text[198:200])}')

Verifying character offsets:

Label: Anstellung
  Start: 0, End: 46
  Extracted start: 'Lernende/r Geb'
  Extracted end:   'üftung EFZ'
  Annotation end:  'üftung EFZ'
  Offset reliable: True

Label: Firmenbeschreibung
  Start: 48, End: 198
  Extracted start: 'Die Hälg Group '
  Extracted end:   'ement. ...'
  Annotation end:  'ent. ...\\n'
  Offset reliable: True

Label: Anstellung
  Start: 200, End: 231
  Extracted start: 'Hälg Group (Leh'
  Extracted end:   'en)\n\nBasel'
  Annotation end:  ')\\n\\nBasel'
  Offset reliable: True

Gap 1 (chars 46-48): '\n\n'
Gap 2 (chars 198-200): '\n\n'


> **Finding:** `value['text']` stores escaped newlines (`\\n`) while `content_clean` contains real newlines. The offsets are correct and reliable. Gaps between spans (`\n\n`) contain no tokens — they are simply skipped by the tokenizer and receive no label.


### 1.4 Dataset-Wide Statistics

In [ ]:
label_counter  = Counter()
total_spans    = 0
empty_entries  = []

for entry in data:
    spans = entry['annotations'][0]['result']
    if len(spans) == 0:
        empty_entries.append(entry)
        continue
    for span in spans:
        label_counter[span['value']['labels'][0]] += 1
        total_spans += 1

lengths     = [len(e['data']['content_clean']) for e in data]
clean_data  = [e for e in data if len(e['annotations'][0]['result']) > 0]
clean_lens  = [len(e['data']['content_clean']) for e in clean_data]

print(f'Raw entries:              {len(data)}')
print(f'Unannotated (removed):    {len(empty_entries)}')
print(f'Clean dataset:            {len(clean_data)}')
print(f'Total labeled spans:      {total_spans}')
print(f'Shortest job ad:          {min(clean_lens)} characters')
print(f'Longest job ad:           {max(clean_lens)} characters')
print(f'Average length:           {sum(clean_lens)//len(clean_lens)} characters')
print(f'\nLabel distribution:')
for label, count in label_counter.most_common():
    pct = count / total_spans * 100
    print(f'  {label:<35s}: {count:5d}  ({pct:.1f}%)')

Raw entries:              2699
Unannotated (removed):    11
Clean dataset:            2688
Total labeled spans:      19666
Shortest job ad:          101 characters
Longest job ad:           19598 characters
Average length:           1591 characters

Label distribution:
  Fähigkeiten und Inhalte            :  5208  (26.5%)
  Anstellung                         :  4001  (20.3%)
  Bewerbungsprozess                  :  2651  (13.5%)
  Abschlüsse                         :  2256  (11.5%)
  Erfahrung                          :  1918  (9.8%)
  Benefits                           :  1391  (7.1%)
  Firmenbeschreibung                 :  1303  (6.6%)
  Firmenkundenbeschreibung           :   769  (3.9%)
  Challenges                         :   108  (0.5%)
  Arbeitsumfeld                      :    61  (0.3%)


> **Finding:** The dataset contains **10 labels**, not 8 as listed in the project specification. `Firmenkundenbeschreibung` and `Arbeitsumfeld` were discovered during exploration and are included in the label mapping.
>
> **Class imbalance:** `Challenges` (0.5%) and `Arbeitsumfeld` (0.3%) together represent less than 1% of all spans. Person 2 must address this using **class-weighted CrossEntropyLoss** during training.


### 1.5 Removed Entries Analysis

In [ ]:
print(f'Removed entries ({len(empty_entries)} total):\n')
for e in empty_entries:
    preview = repr(e['data']['content_clean'][:80])
    print(f"  ID {e['id']:5d} | length {len(e['data']['content_clean']):6d} chars | {preview}")

Removed entries (11 total):

  ID   431 | length   8101 chars | 'Employé(-e) de restaurants - responsable laveries La Coupole et Chariots du caf'
  ID   765 | length   2684 chars | 'Contexte\n\nLa ville de Lausanne est en pleine mutation. Ces prochaines années, p'
  ID   779 | length   1715 chars | "Aujourd'hui, vous allez vivre pleinement votre expérience professionnelle\n\nMont"
  ID   786 | length   3430 chars | '11.01.2023 Genève Plainpalais\nLieu\n\nGenève\n\nVotre profil\n\nCFC de monteur-élect'
  ID   813 | length  22980 chars | 'BTC Business Analyst – Marketing %26 Sales Team – Young Graduate position\n%3CDIV'
  ID  3604 | length   3057 chars | "Lieu du travail: Clinique Cecil | Lausanne\nOccupation par: pour une date d'entr"
  ID  3859 | length  54935 chars | 'ftlx0!|!ftlUtil_resetPage!%24!requisitionDescriptionInterface!|!descRequisition!'
  ID  3885 | length    218 chars | 'Fachleute aus der grafischen IndustrieGrafische Branche | javascript:__doPostBa'
  ID  4614 | length  

> **Removal categories:**
> - **French-language ads** (IDs 431, 765, 779, 786, 3604): No annotations present. Training on these would incorrectly label all tokens as `O`. Note: French language alone is not the reason — `bert-base-multilingual-cased` supports French. The missing annotations are.
> - **HTML/URL-encoded text** (IDs 813, 4614): Unreadable content (`%26`, `%3CDIV`) from incomplete crawler cleaning.
> - **Corrupted entries** (IDs 3859, 3885, 5047, 5261): System error pages or crawler artifacts.


---
## Section 2 — Tokenization

### 2.1 Why BertTokenizerFast?

The **Fast** variant is a hard requirement — not optional. Only `BertTokenizerFast` provides `offset_mapping`, which maps each token back to its character position in the original text. Without it, label alignment is impossible.

Additional reasons:
- **Multilingual-cased**: handles German umlauts (ä, ö, ü, ß) and preserves capitalisation (meaningful in German — all nouns are capitalised)
- `add_special_tokens=False` during preprocessing: `[CLS]` and `[SEP]` are not added per-chunk, keeping offset mapping clean


In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained(
    'bert-base-multilingual-cased',
    use_fast=True
)

print(f'Tokenizer type: {type(tokenizer).__name__}')
print(f'Is fast:        {tokenizer.is_fast}')

Tokenizer type: BertTokenizer
Is fast:        True


> **Note:** The class name displays as `BertTokenizer` but `is_fast: True` confirms the fast backend is active and `offset_mapping` is available.


### 2.2 Tokenizing an Example

In [ ]:
text_sample = (
    'Lernende/r Gebäudetechnikplaner/in Lüftung EFZ\n\n'
    'Die Hälg Group ist ein Schweizer Familienunternehmen.'
)

encoding = tokenizer(
    text_sample,
    return_offsets_mapping=True,
    add_special_tokens=False
)

tokens  = encoding['input_ids']
offsets = encoding['offset_mapping']

print(f'Number of tokens: {len(tokens)}\n')
for i, (token_id, (cs, ce)) in enumerate(zip(tokens, offsets)):
    token_text = tokenizer.decode([token_id])
    print(f'Token {i:3d}: {repr(token_text):25s} (chars {cs:3d}–{ce:3d})')

Number of tokens: 28

Token   0: 'Le'                      (chars   0–  2)
Token   1: '##rnen'                  (chars   2–  6)
Token   2: '##de'                    (chars   6–  8)
Token   3: '/'                       (chars   8–  9)
Token   4: 'r'                       (chars   9– 10)
Token   5: 'Gebäude'                 (chars  11– 18)
Token   6: '##technik'               (chars  18– 25)
Token   7: '##plane'                 (chars  25– 30)
Token   8: '##r'                     (chars  30– 31)
Token   9: '/'                       (chars  31– 32)
Token  10: 'in'                      (chars  32– 34)
Token  11: 'L'                       (chars  35– 36)
Token  12: '##ü'                     (chars  36– 37)
Token  13: '##ft'                    (chars  37– 39)
Token  14: '##ung'                   (chars  39– 42)
Token  15: 'EF'                      (chars  43– 45)
Token  16: '##Z'                     (chars  45– 46)
Token  17: 'Die'                     (chars  48– 51)
Token  18: 'H'          

> **Sub-word tokenization:** The `##` prefix marks a continuation token. For example, `Lernende` is split into `Le` + `##rnen` + `##de`. This means one word can produce multiple tokens — each needing its own label. This is the core challenge that `find_token_span()` solves.
>
> **Gap handling:** Tokens 16 (`##Z`, chars 45–46) and 17 (`Die`, chars 48–51) skip chars 46–48 (`\n\n`). The tokenizer produces no token for whitespace — those character positions simply have no corresponding token.


---
## Section 3 — Label Alignment Verification

### 3.1 Full Label Alignment on Entry 4163

For each token, we check whether its character range falls inside any annotation span. If yes, it receives that label. If no span matches, it receives `O`.


In [ ]:
entry = next(item for item in data if item['id'] == 4163)
text  = entry['data']['content_clean']
spans = entry['annotations'][0]['result']

encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    add_special_tokens=False
)
tokens  = encoding['input_ids']
offsets = encoding['offset_mapping']

# Initialize all as O
labels = ['O'] * len(tokens)

# Apply each annotation span
for span in spans:
    v          = span['value']
    char_start = v['start']
    char_end   = v['end']
    label      = v['labels'][0]
    for i, (tok_start, tok_end) in enumerate(offsets):
        if tok_start >= char_start and tok_end <= char_end:
            labels[i] = label

print(f'Total tokens: {len(tokens)}\n')
for i, (token_id, (cs, ce), label) in enumerate(zip(tokens, offsets, labels)):
    tok = tokenizer.decode([token_id])
    print(f'Token {i:3d}: {repr(tok):20s} (chars {cs:3d}–{ce:3d})  →  {label}')

Total tokens: 59

Token   0: 'Le'                 (chars   0–  2)  →  Anstellung
Token   1: '##rnen'             (chars   2–  6)  →  Anstellung
Token   2: '##de'               (chars   6–  8)  →  Anstellung
Token   3: '/'                  (chars   8–  9)  →  Anstellung
Token   4: 'r'                  (chars   9– 10)  →  Anstellung
Token   5: 'Gebäude'            (chars  11– 18)  →  Anstellung
Token   6: '##technik'          (chars  18– 25)  →  Anstellung
Token   7: '##plane'            (chars  25– 30)  →  Anstellung
Token   8: '##r'                (chars  30– 31)  →  Anstellung
Token   9: '/'                  (chars  31– 32)  →  Anstellung
Token  10: 'in'                 (chars  32– 34)  →  Anstellung
Token  11: 'L'                  (chars  35– 36)  →  Anstellung
Token  12: '##ü'               (chars  36– 37)  →  Anstellung
Token  13: '##ft'               (chars  37– 39)  →  Anstellung
Token  14: '##ung'              (chars  39– 42)  →  Anstellung
Token  15: 'EF'                 (chars

> **Verification:** All 59 tokens are correctly labeled:
> - Tokens 0–16 (chars 0–46) → `Anstellung` ✅
> - Tokens 17–48 (chars 48–198) → `Firmenbeschreibung` ✅
> - Tokens 49–58 (chars 200–231) → `Anstellung` ✅
> - Gaps at chars 46–48 and 198–200 (`\n\n`) produce no tokens — correctly skipped ✅
> - No `O` tokens in this entry — the annotations cover the entire text ✅


### 3.2 Verifying find_token_span() Boundaries

In [ ]:
print('Verifying find_token_span() for each annotation:\n')
for span in spans:
    v          = span['value']
    char_start = v['start']
    char_end   = v['end']
    label      = v['labels'][0]

    start_token = end_token = None
    fallback_used = []

    for i, (ts, te) in enumerate(offsets):
        if start_token is None and ts <= char_start < te:
            start_token = i
        if end_token is None and ts < char_end <= te:
            end_token = i
        if start_token and end_token:
            break

    if start_token is None:
        start_token = min(range(len(offsets)), key=lambda i: abs(offsets[i][0] - char_start))
        fallback_used.append('START')
    if end_token is None:
        end_token = min(range(len(offsets)), key=lambda i: abs(offsets[i][1] - char_end))
        fallback_used.append('END')

    fallback_str = f' ⚠️ Fallback used: {fallback_used}' if fallback_used else ' ✅ Exact match'
    print(f'Label: {label}')
    print(f'  Char span:   {char_start}–{char_end}')
    print(f'  Token span:  {start_token}–{end_token}')
    print(f'  First token: {repr(tokenizer.decode([tokens[start_token]]))} (chars {offsets[start_token][0]}–{offsets[start_token][1]})')
    print(f'  Last token:  {repr(tokenizer.decode([tokens[end_token]]))} (chars {offsets[end_token][0]}–{offsets[end_token][1]})')
    print(f'  Boundary:{fallback_str}\n')

Verifying find_token_span() for each annotation:

Label: Anstellung
  Char span:   0–46
  Token span:  0–16
  First token: 'Le' (chars 0–2)
  Last token:  '##Z' (chars 45–46)
  Boundary: ✅ Exact match

Label: Firmenbeschreibung
  Char span:   48–198
  Token span:  17–48
  First token: 'Die' (chars 48–51)
  Last token:  '.' (chars 197–198)
  Boundary: ✅ Exact match

Label: Anstellung
  Char span:   200–231
  Token span:  49–58
  First token: 'H' (chars 200–201)
  Last token:  'Basel' (chars 226–231)
  Boundary: ✅ Exact match


> **All boundaries found exactly — no fallback needed for this entry.** The fallback mechanism exists for cases where annotation boundaries land on whitespace or between sub-word tokens. It finds the nearest token rather than losing the annotation entirely.


---
## Section 4 — Sliding Window Verification

### 4.1 Why a Sliding Window?

BERT has a hard maximum of **512 tokens** per input. Approximately **25.6% of job ads** (~687 entries) exceed this limit. Simply truncating would lose annotations in the later parts of those ads.

The sliding window splits long documents into overlapping chunks:
- `window_size = 510` tokens per chunk
- `stride = 256` tokens (how far the window moves each step)
- Overlap = `window_size - stride = 254` tokens

The overlap ensures tokens near chunk boundaries appear with full surrounding context in at least one chunk.


### 4.2 Finding a Long Entry

In [ ]:
# Find entry 3769 — 513 tokens, just 1 over the limit
entry_long = next(item for item in data if item['id'] == 3769)
text_long  = entry_long['data']['content_clean']
spans_long = entry_long['annotations'][0]['result']

print(f'Text length: {len(text_long)} characters')
print(f'Number of annotations: {len(spans_long)}')
print(f'\nAnnotations:')
for span in spans_long:
    v = span['value']
    print(f"  chars {v['start']:4d}–{v['end']:4d}  →  {v['labels'][0]}")

Text length: 2162 characters
Number of annotations: 15

Annotations:
  chars  222– 371  →  Anstellung
  chars   95– 136  →  Bewerbungsprozess
  chars  137– 220  →  Firmenbeschreibung
  chars    0–  95  →  Anstellung
  chars  373– 417  →  Fähigkeiten und Inhalte
  chars  418– 434  →  Erfahrung
  chars  435– 477  →  Fähigkeiten und Inhalte
  chars  479– 679  →  Anstellung
  chars  524– 894  →  Fähigkeiten und Inhalte
  chars  896–1262  →  Benefits
  chars 1264–1407  →  Abschlüsse
  chars 1409–1530  →  Fähigkeiten und Inhalte
  chars 1532–1775  →  Benefits
  chars 1777–2132  →  Bewerbungsprozess
  chars 2134–2162  →  Anstellung


> **Two findings from this entry's annotations:**
> - **Overlapping spans:** chars 524–679 are covered by both `Anstellung` (479–679) and `Fähigkeiten und Inhalte` (524–894). The last-processed label wins — a minor annotation inconsistency.
> - **Unsorted annotations:** Spans are not in character order in the JSON. The code handles this correctly since it checks each token against all spans independently.


### 4.3 Running the Sliding Window

In [ ]:
# Full sliding window pipeline on entry 3769
encoding_long = tokenizer(
    text_long,
    return_offsets_mapping=True,
    add_special_tokens=False
)
full_tokens  = encoding_long['input_ids']
full_offsets = encoding_long['offset_mapping']

# Initialize labels
full_labels = ['O'] * len(full_tokens)

# Apply annotations
for span in spans_long:
    v = span['value']
    for i, (ts, te) in enumerate(full_offsets):
        if ts >= v['start'] and te <= v['end']:
            full_labels[i] = v['labels'][0]

print(f'Total tokens: {len(full_tokens)}')
print(f'Unique labels: {set(full_labels)}')

# Sliding window
window_size = 510
stride      = 256
chunks      = []

for chunk_start in range(0, len(full_tokens), stride):
    chunk_end = min(chunk_start + window_size, len(full_tokens))
    chunks.append((full_tokens[chunk_start:chunk_end], full_labels[chunk_start:chunk_end]))
    if chunk_end == len(full_tokens):
        break

print(f'\nNumber of chunks produced: {len(chunks)}')
for i, (ctok, clab) in enumerate(chunks):
    start = i * stride
    print(f'\nChunk {i+1}: {len(ctok)} tokens (positions {start}–{start + len(ctok)})')
    print(f'  Labels present: {set(clab)}')
    print(f'  First token: {repr(tokenizer.decode([ctok[0]]))}')
    print(f'  Last token:  {repr(tokenizer.decode([ctok[-1]]))}')

print(f'\nOverlap verification:')
print(f'  Token 256 same in chunk 1 and chunk 2: {chunks[0][0][256] == chunks[1][0][0]}')

Total tokens: 513
Unique labels: {'Erfahrung', 'Firmenbeschreibung', 'Bewerbungsprozess', 'Fähigkeiten und Inhalte', 'Benefits', 'O', 'Abschlüsse', 'Anstellung'}

Number of chunks produced: 2

Chunk 1: 510 tokens (positions 0–510)
  Labels present: {'Erfahrung', 'Firmenbeschreibung', 'Bewerbungsprozess', 'Fähigkeiten und Inhalte', 'Benefits', 'O', 'Abschlüsse', 'Anstellung'}
  First token: 'Praxis'
  Last token:  '##ha'

Chunk 2: 257 tokens (positions 256–513)
  Labels present: {'Bewerbungsprozess', 'Fähigkeiten und Inhalte', 'Benefits', 'Abschlüsse', 'Anstellung'}
  First token: '##fi'
  Last token:  'GmbH'

Overlap verification:
  Token 256 same in chunk 1 and chunk 2: True


> **Sliding window verified:**
> - 513 tokens → 2 chunks ✅
> - Chunk 1: tokens 0–510 (510 tokens, full window) ✅
> - Chunk 2: tokens 256–513 (257 tokens, remainder) ✅
> - Overlap: tokens 256–510 appear in both chunks ✅
> - All labels preserved across both chunks ✅
> - Sub-word split at boundary (`##ha` / `##fi`) is normal — the full word appears intact within at least one chunk ✅


---
## Section 5 — Full Pipeline Sanity Check

### 5.1 Run preprocessing.py on a Small Subset

Before running the full pipeline on all 2,688 job ads (which takes significant time), we verify correctness on a 20-entry subset.

The full run is executed on Day 3 and produces:
- `../data/train_dataset.pt`
- `../data/test_dataset.pt`  
- `../model/id2label.json`


In [ ]:
import torch
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split

# Use a 20-entry subset for fast verification
subset = [e for e in data if len(e['annotations'][0]['result']) > 0][:20]

label2id = {'O': 0}
processed_data = []

for entry in subset:
    text_e  = entry['data']['content_clean']
    spans_e = entry['annotations'][0]['result']

    enc = tokenizer(text_e, return_offsets_mapping=True, add_special_tokens=False)
    full_toks = enc['input_ids']
    full_offs = enc['offset_mapping']
    full_labs = ['O'] * len(full_toks)

    for span in spans_e:
        v = span['value']
        for i, (ts, te) in enumerate(full_offs):
            if ts >= v['start'] and te <= v['end']:
                full_labs[i] = v['labels'][0]

    # Sliding window
    for chunk_start in range(0, len(full_toks), 256):
        chunk_end = min(chunk_start + 510, len(full_toks))
        ctoks = full_toks[chunk_start:chunk_end]
        clabs = full_labs[chunk_start:chunk_end]

        # Build label2id
        for lab in clabs:
            if lab not in label2id:
                label2id[lab] = len(label2id)

        processed_data.append((ctoks, [label2id[l] for l in clabs]))
        if chunk_end == len(full_toks):
            break

print(f'Subset entries processed: {len(subset)}')
print(f'Total chunks produced:    {len(processed_data)}')
print(f'Labels discovered:        {list(label2id.keys())}')

Subset entries processed: 20
Total chunks produced:    22
Labels discovered:        ['O', 'Anstellung', 'Firmenbeschreibung', 'Bewerbungsprozess', 'Fähigkeiten und Inhalte', 'Erfahrung', 'Abschlüsse', 'Benefits', 'Firmenkundenbeschreibung']


### 5.2 Verify TensorDataset Shape and Padding

In [ ]:
from torch.nn.utils.rnn import pad_sequence

train_data, test_data = train_test_split(processed_data, test_size=0.2, random_state=42)

def make_dataset(data):
    input_ids  = [torch.tensor(d[0]) for d in data]
    labels     = [torch.tensor(d[1]) for d in data]
    attn_masks = [torch.ones_like(t) for t in input_ids]
    return TensorDataset(
        pad_sequence(input_ids,  batch_first=True, padding_value=0),
        pad_sequence(labels,     batch_first=True, padding_value=-100),
        pad_sequence(attn_masks, batch_first=True, padding_value=0)
    )

train_dataset = make_dataset(train_data)
test_dataset  = make_dataset(test_data)

print(f'Train dataset: {len(train_dataset)} chunks')
print(f'Test dataset:  {len(test_dataset)} chunks')
print(f'\nTensor shapes (train):')
print(f'  input_ids:      {train_dataset.tensors[0].shape}')
print(f'  labels:         {train_dataset.tensors[1].shape}')
print(f'  attention_mask: {train_dataset.tensors[2].shape}')
print(f'\nPadding check:')
print(f'  Unique label padding values: {train_dataset.tensors[1].unique()[-3:]}')
print(f'  -100 present (correct padding): {-100 in train_dataset.tensors[1]}')

Train dataset: 17 chunks
Test dataset:  5 chunks

Tensor shapes (train):
  input_ids:      torch.Size([17, 510])
  labels:         torch.Size([17, 510])
  attention_mask: torch.Size([17, 510])

Padding check:
  Unique label padding values: tensor([-100,    0,    1])
  -100 present (correct padding): True


> **All checks pass:**
> - Correct number of chunks after 80/20 split ✅
> - All three tensors have matching shapes ✅
> - Label padding uses `-100` (not `label2id['O']`) ✅
> - `-100` confirmed present in label tensors ✅

---

## Summary

| Verification | Result |
|---|---|
| Character offsets reliable in annotated.json | ✅ |
| Dataset contains 10 labels (not 8 as in spec) | ✅ |
| 11 unannotated entries correctly identified and categorized | ✅ |
| BertTokenizerFast loads with offset_mapping | ✅ |
| Sub-word tokenization of German text correct | ✅ |
| Label alignment assigns correct labels to all tokens | ✅ |
| Gaps between spans handled correctly | ✅ |
| find_token_span finds exact boundaries | ✅ |
| Sliding window produces correct chunks | ✅ |
| Overlap between chunks verified | ✅ |
| TensorDataset shapes correct | ✅ |
| Label padding uses -100 | ✅ |

**The preprocessing pipeline is verified and ready for full execution on Day 3.**
